# 07 · Shallow baselines: character n-gram TF-IDF and stylometric features (logistic regression)
**Purpose:** a cheap, transparent detector trained on the **same experiment specs** as notebook 06 (E1, E2, E3, optional E6). Character n-grams are script-sensitive, so this baseline is diagnostic for the family-vs-script question (RQ2) and for homoglyph attacks (RQ4).

**v1.2:** adds detector `stylo` — punctuation densities + mean sentence length (the features of the 04 audit). It measures how far a *shallow style* signal goes, in-language and cross-lingually; neural detectors must beat it.

Deterministic ⇒ one run per spec (seed recorded as the first config seed). `C` is selected on the validation AUROC.
**Runtime:** CPU (high-RAM runtime helps), ~1–2 min per spec.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "07_ngram_baseline"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); _ = env_snapshot(NOTEBOOK)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
NG = CFG["ngram"]
RUN_EXPS = ["E2", "E3", "E1"]            # add "E6" if the augmentation arm is run
FULL_EVAL = {("E2", "all"), ("E6", "all-aug")}
CORE_SETS = {"val", "test", "ext_coling_raw", "ext_coling_norm"}
ds = load_dataset(); EVAL = load_eval_texts()
seed = SEEDS[0]

for exp in RUN_EXPS:
    for spec in CFG["experiments"][exp]:
        rid = make_run_id(exp, "ngram", spec["name"], spec.get("heldout_family") or "none", seed)
        if is_done(rid):
            continue
        t0 = time.time()
        register_run(rid, "running", exp=exp, detector="ngram", train=spec["name"],
                     heldout=spec.get("heldout_family") or "none", seed=seed, params_json=json.dumps({"spec": spec, **NG}))
        tr, va = training_frame(ds, spec)
        if spec.get("augment"):
            tr = augment_train(tr, spec["augment"], seed)
        vec = TfidfVectorizer(analyzer=NG["analyzer"], ngram_range=tuple(NG["ngram_range"]), max_features=NG["max_features"],
                              sublinear_tf=True, min_df=2, lowercase=False)
        Xtr = vec.fit_transform(tr.text); Xva = vec.transform(va.text)
        best = None
        for C in NG["C_grid"]:
            clf = LogisticRegression(C=C, max_iter=3000, solver="liblinear", random_state=seed).fit(Xtr, tr.label)
            a = auroc(va.label.values, clf.decision_function(Xva))
            if best is None or a > best[0]:
                best = (a, C, clf)
        sets = set(EVAL.eval_set.unique()) if (exp, spec["name"]) in FULL_EVAL else CORE_SETS
        ev = EVAL[EVAL.eval_set.isin(sets)]
        sc = best[2].decision_function(vec.transform(ev.text))
        save_parquet(pd.DataFrame({"doc_id": ev.doc_id.values, "score": sc, "prob": 1 / (1 + np.exp(-sc))}), pred_path(rid))
        register_run(rid, "done", seconds=round(time.time() - t0, 1),
                     notes=json.dumps({"n_train": len(tr), "best_C": best[1], "val_auroc": round(best[0], 4)}))
        log.info(f"{rid}: C={best[1]} val AUROC={best[0]:.4f}")

## Stylometric baseline (`stylo`)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
STY_PUNCT = list(".,;:!?—-\"'()[]/%…")
def stylo_feats(texts):
    rows = []
    for t in texts:
        w = max(1, n_words(t)); sl = [n_words(x) for x in _SENT_END.split(t) if x.strip()] or [0]
        rows.append([t.count(c) / w for c in STY_PUNCT] + [np.mean(sl), np.std(sl), np.mean([len(x) for x in t.split()] or [0])])
    return np.array(rows)

for exp in RUN_EXPS:
    for spec in CFG["experiments"][exp]:
        rid = make_run_id(exp, "stylo", spec["name"], spec.get("heldout_family") or "none", seed)
        if is_done(rid):
            continue
        t0 = time.time()
        register_run(rid, "running", exp=exp, detector="stylo", train=spec["name"],
                     heldout=spec.get("heldout_family") or "none", seed=seed, params_json=json.dumps({"spec": spec, "features": STY_PUNCT + ["sent_mean", "sent_std", "word_len"]}))
        tr, va = training_frame(ds, spec)
        if spec.get("augment"):
            tr = augment_train(tr, spec["augment"], seed)
        Xtr, Xva = stylo_feats(tr.text), stylo_feats(va.text)
        best = None
        for C in NG["C_grid"]:
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000, random_state=seed)).fit(Xtr, tr.label)
            a = auroc(va.label.values, clf.decision_function(Xva))
            if best is None or a > best[0]:
                best = (a, C, clf)
        sets = set(EVAL.eval_set.unique()) if (exp, spec["name"]) in FULL_EVAL else CORE_SETS
        ev = EVAL[EVAL.eval_set.isin(sets)]
        sc = best[2].decision_function(stylo_feats(ev.text))
        save_parquet(pd.DataFrame({"doc_id": ev.doc_id.values, "score": sc, "prob": 1 / (1 + np.exp(-sc))}), pred_path(rid))
        register_run(rid, "done", seconds=round(time.time() - t0, 1),
                     notes=json.dumps({"n_train": len(tr), "best_C": best[1], "val_auroc": round(best[0], 4)}))
        log.info(f"{rid}: C={best[1]} val AUROC={best[0]:.4f}")